### Test Preprocessing Components ###

This notebook validates individual preprocessing components (Imputer, Scaler, Encoder) to ensure transformations match expectations.

**1. Test Median Imputer:**

In [1]:
import pandas as pd
from sklearn.impute import SimpleImputer
import numpy as np

# Sample data with missing values
data = pd.DataFrame({"age": [25, np.nan, 40, 35, np.nan]})

# Fit imputer
imputer = SimpleImputer(strategy="median")
imputed = imputer.fit_transform(data)

# Convert back to DataFrame
imputed_df = pd.DataFrame(imputed, columns=["age"])

print("Original:", data.values.ravel())
print("Imputed:", imputed_df.values.ravel())

# ✅ Assert median imputation
assert imputer.statistics_[0] == 35, "Median should be 35"
assert not imputed_df.isnull().any().any(), "No missing values should remain"

Original: [25. nan 40. 35. nan]
Imputed: [25. 35. 40. 35. 35.]


**Output Data (after imputation)**
| age |
|-----|
| 25  |
| 35  |
| 40  |
| 35  |
| 35  |

**Validation:**
- Median of non‑missing values = **35**  
- All NaNs replaced → ✅

---

**2. Test Standard Scaler:**

In [6]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Sample numeric data
data = pd.DataFrame({"distance_to_clinic_km": [2, 4, 6, 8]})

scaler = StandardScaler()
scaled = scaler.fit_transform(data)

scaled_df = pd.DataFrame(scaled, columns=["distance_to_clinic_km"])

print("Original:", data.values.ravel())
print("Scaled:", scaled_df.values.ravel())

# ✅ Assert mean ~ 0 and std ~ 1
mean_val = scaled_df["distance_to_clinic_km"].mean()
std_val = scaled_df["distance_to_clinic_km"].std(ddof=0)  # population std

assert abs(mean_val) < 1e-6, f"Mean should be ~0, got {mean_val}"
assert abs(std_val - 1) < 1e-6, f"Std should be ~1, got {std_val}"

Original: [2 4 6 8]
Scaled: [-1.34164079 -0.4472136   0.4472136   1.34164079]


**Output Data (scaled)**
| distance_to_clinic_km |
| --- |
| -1.3416 |
| -0.4472 |
| 0.4472 |
| 1.3416 |

**Validation:**
- Mean ≈ 0  
- Std ≈ 1 → ✅

**3. Test One‑Hot Encoder:**

In [7]:
from sklearn.preprocessing import OneHotEncoder

# Sample categorical data
data = pd.DataFrame({"appointment_type": ["General", "Follow-up", "Specialist", "General"]})

encoder = OneHotEncoder(sparse_output=False)
encoded = encoder.fit_transform(data)

encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(["appointment_type"]))

print("Original:", data.values.ravel())
print("Encoded:\n", encoded_df)

# ✅ Assert correct number of columns
assert encoded_df.shape[1] == len(encoder.categories_[0]), "Should match unique categories"

Original: ['General' 'Follow-up' 'Specialist' 'General']
Encoded:
    appointment_type_Follow-up  appointment_type_General  \
0                         0.0                       1.0   
1                         1.0                       0.0   
2                         0.0                       0.0   
3                         0.0                       1.0   

   appointment_type_Specialist  
0                          0.0  
1                          0.0  
2                          1.0  
3                          0.0  


**Output Data (encoded)**
| appointment_type_Follow-up | appointment_type_General | appointment_type_Specialist |
|-----------------------------|--------------------------|-----------------------------|
| 0                           | 1                        | 0                           |
| 1                           | 0                        | 0                           |
| 0                           | 0                        | 1                           |
| 0                           | 1                        | 0                           |

**Validation:**
- 3 unique categories → 3 binary columns  
- Encoding matches input → ✅

**Summary**
- Median imputer correctly replaces missing values.  
- StandardScaler normalizes numeric features to mean 0, std 1.  
- OneHotEncoder expands categorical features into binary flags.  

All preprocessing components validated successfully.